In [7]:
import os
from pathlib import Path
import sys
import shutil
import easyocr
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [8]:
#  Add project root to sys.path for local imports
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [9]:
from src.configs.settings import settings

In [5]:
# CONFIGURATION
INPUT_DIR = settings.PLATE_OUTPUT_DIR / "images" 
OUTPUT_BASE = settings.LICENSE_INPUT_DIR
TRAIN_RATIO = 0.8
IMAGE_EXTS = ('.jpg', '.jpeg', '.png')

# Initialize EasyOCR Reader (English)
reader = easyocr.Reader(['en'])

# Create folder structure
for split in ['train', 'val']:
    os.makedirs(os.path.join(OUTPUT_BASE, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_BASE, split, 'labels'), exist_ok=True)

print("Folders created. Starting OCR labeling...")

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Folders created. Starting OCR labeling...


In [ ]:
# all_data = []
# raw_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(IMAGE_EXTS)]

# for idx, filename in enumerate(tqdm(raw_files)):
#     img_path = os.path.join(INPUT_DIR, filename)
    
#     # Perform OCR
#     result = reader.readtext(img_path)
    
#     # Extract and clean the text
#     # We join all detected text blocks and keep only Alphanumeric
#     text = "".join([res[1] for res in result]).upper()
#     text = "".join(filter(str.isalnum, text))
    
#     if not text:
#         text = "UNKNOWN" # Placeholder for manual fix later
    
#     # Store info for splitting
#     new_name = f"{idx:04d}" # e.g., 0001
#     all_data.append({
#         'old_path': img_path,
#         'new_name': new_name,
#         'label': text
#     })

# print(f"Processed {len(all_data)} images.")

  0%|          | 0/4790 [00:00<?, ?it/s]c:\Users\Samuel.Ozechi\Downloads\projects\vision-based-access-intelligence\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  0%|          | 1/4790 [00:00<41:19,  1.93it/s]c:\Users\Samuel.Ozechi\Downloads\projects\vision-based-access-intelligence\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  0%|          | 2/4790 [00:00<24:19,  3.28it/s]c:\Users\Samuel.Ozechi\Downloads\projects\vision-based-access-intelligence\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  0%|          | 3

Processed 4790 images.


In [ ]:
# # Split into Train/Validation Sets
# train_data, val_data = train_test_split(all_data, test_size=(1 - TRAIN_RATIO), random_state=42)

# def save_split(data_list, split_name):
#     for item in data_list:
#         # Define paths
#         target_img_dir = os.path.join(OUTPUT_BASE, split_name, 'images')
#         target_lbl_dir = os.path.join(OUTPUT_BASE, split_name, 'labels')
        
#         # Copy and rename image
#         shutil.copy(item['old_path'], os.path.join(target_img_dir, f"{item['new_name']}.jpg"))
        
#         # Save label text file
#         with open(os.path.join(target_lbl_dir, f"{item['new_name']}.txt"), 'w') as f:
#             f.write(item['label'])

# save_split(train_data, 'train')
# save_split(val_data, 'val')

# print(f"Dataset finalized! Find your data in the '{OUTPUT_BASE}' folder.")

Dataset finalized! Find your data in the 'C:\Users\Samuel.Ozechi\Downloads\projects\vision-based-access-intelligence\data\license_recognition\inputs' folder.


### **Using OpenAI LLM**

In [10]:
import os
import base64
import json
import shutil
from openai import OpenAI
from tqdm import tqdm
from sklearn.model_selection import train_test_split

client = OpenAI(base_url="https://ai-gateway.isw.la/v1", api_key="sk-0iDc3wvNLug028z6TQt20w")

# Configuration
BATCH_SIZE = 10  # Number of images to send in one API call

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')


In [12]:
raw_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
all_results = []

for i in tqdm(range(0, len(raw_files), BATCH_SIZE)):
    batch_files = raw_files[i : i + BATCH_SIZE]
    content = [
        {
            "type": "text", 
            "text": "Identify the license plate numbers in these images. If a plate is not clearly visible or recognizable, return 'UNKNOWN' for that image. Return ONLY a JSON object where keys are the image index (0, 1, 2...) and values are the plate strings or 'UNKNOWN'. Example: {'0': 'ABC-123DE', '1': 'UNKNOWN'}. NOTE THAT: Nigerian plates usually follow AAA-111AA format. "
        }
    ]
    
    # Attach images to the batch request
    for file in batch_files:
        base64_image = encode_image(os.path.join(INPUT_DIR, file))
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{base64_image}", "detail": "high"}
        })

    try:
        response = client.chat.completions.create(
            model="gpt-4.1", # Use mini for cost-efficiency, gpt-4o for max accuracy
            messages=[{"role": "user", "content": content}],
            response_format={ "type": "json_object" }
        )
        
        batch_json = json.loads(response.choices[0].message.content)
        
        # Store results with original paths, skipping UNKNOWN labels
        batch_kept = 0
        for idx, plate_text in batch_json.items():
            file_idx = int(idx)
            if file_idx < len(batch_files):
                label = str(plate_text).upper().strip()
                if label != "UNKNOWN":
                    all_results.append({
                        'old_path': os.path.join(INPUT_DIR, batch_files[file_idx]),
                        'label': label
                    })
                    batch_kept += 1
        
        print(f"Batch {i//BATCH_SIZE + 1}: {batch_kept}/{len(batch_files)} images kept (UNKNOWN skipped)")
    except Exception as e:
        print(f"Error in batch {i//BATCH_SIZE + 1}: {e}")

print(f"Successfully labeled {len(all_results)} images.")

  0%|          | 0/479 [00:00<?, ?it/s]

  0%|          | 1/479 [00:07<1:00:18,  7.57s/it]

Batch 1: 10/10 images kept (UNKNOWN skipped)


  0%|          | 2/479 [00:11<44:52,  5.64s/it]  

Batch 2: 9/10 images kept (UNKNOWN skipped)


...

Batch 477: 10/10 images kept (UNKNOWN skipped)


100%|█████████▉| 478/479 [43:54<00:04,  4.90s/it]

Batch 478: 7/10 images kept (UNKNOWN skipped)


100%|██████████| 479/479 [43:59<00:00,  5.51s/it]

Batch 479: 10/10 images kept (UNKNOWN skipped)
Successfully labeled 4522 images.


In [14]:
all_results[0]  # Show sample results

{'old_path': 'C:\\Users\\Samuel.Ozechi\\Downloads\\projects\\vision-based-access-intelligence\\data\\plate_detection\\outputs\\images\\1000_jpg.rf.1d2e55ae78d46b7bc2a2b3d692c06562_plate_refined.jpg',
 'label': 'SWK-4E5HB'}

In [ ]:
train_data, val_data = train_test_split(all_results, test_size=(1 - TRAIN_RATIO), random_state=42)

def save_and_rename(data_list, split_name):
    for i, item in enumerate(data_list):
        new_name = f"{i:04d}"
        target_img = os.path.join(OUTPUT_BASE, split_name, 'images', f"{new_name}.jpg")
        target_lbl = os.path.join(OUTPUT_BASE, split_name, 'labels', f"{new_name}.txt")
        
        os.makedirs(os.path.dirname(target_img), exist_ok=True)
        os.makedirs(os.path.dirname(target_lbl), exist_ok=True)
        
        shutil.copy(item['old_path'], target_img)
        with open(target_lbl, 'w') as f:
            f.write(item['label'])

save_and_rename(train_data, 'train')
save_and_rename(val_data, 'val')

print("Process Complete!")

Process Complete!


: 